# Phonotactic-Dependency Distance Minimization Across UD Treebanks

This notebook demonstrates the evaluation pipeline for the **phonotactic constraint hypothesis**: languages with higher phonological density (more phonemes, more complex phonotactics) should exhibit stronger dependency distance minimization (DLM), as speakers need to keep related words closer together to compensate for the difficulty of parsing longer phonological sequences.

**Metrics computed:**
1. Mean Dependency Distance (MDD) in word-space and phoneme-space
2. Phoneme-Position Adjoining Distance (PPAD) as the phoneme-space analog of MDD
3. DLM strength metric (inverse of MDD) for each language
4. Functional vs lexical dependency distance split
5. Shannon entropy of dependency distance distributions

**Regression analysis:**
- OLS regression of MDD on phonological density with word-order typology as covariate
- Mixed-effects model (when sufficient data) with language family as random effect
- Spearman rank correlation as non-parametric robustness check

Data sourced from synthetic treebanks (the framework supports loading real UD treebanks from `commul/universal_dependencies` on HuggingFace when the `datasets` library is available).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'statsmodels==0.14.6', 'matplotlib==3.10.0', 'pandas==2.2.2')

In [ ]:
import json
import math
import resource
import sys
import gc
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict

import numpy as np
from loguru import logger
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

# Setup logging (notebook-friendly)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

try:
    from scipy import stats
    from scipy.special import entr
except ImportError:
    logger.warning("scipy not available, using numpy fallbacks")
    stats = None
    entr = None

try:
    import statsmodels.api as sm
    from statsmodels.regression.mixed_linear_model import MixedLM
except ImportError:
    logger.warning("statsmodels not available, using numpy OLS fallback")
    sm = None
    MixedLM = None

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-45370e-phonotactic-constraint-on-dependency/main/round-1/evaluation-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['languages'])} languages")
print(f"Languages: {[l['lang_code'] for l in data['languages']]}")

## Configuration

All tunable parameters are defined here. Start with minimum values for a fast demo run.

In [ ]:
# Configuration parameters
# Original values: n_languages=12, n_sentences=50, avg_words=10
N_LANGUAGES = 10          # Number of languages to process (min: 3, original: 12)
N_SENTENCES = 20          # Sentences per synthetic treebank (min: 5, original: 50)
AVG_WORDS = 10            # Average words per sentence (min: 10, original: 10)
RANDOM_SEED = 42          # Random seed for reproducibility

print(f"Config: {N_LANGUAGES} languages, {N_SENTENCES} sentences, {AVG_WORDS} avg words")

## Core Computation Functions

These functions compute dependency distances, phoneme-space distances, and phonological density estimates. Code copied from `eval.py` with minimal changes.

In [ ]:
def compute_dependency_distances(treebank_data: List[Dict]) -> Dict[str, float]:
    """Compute mean dependency distance and distribution statistics."""
    if not treebank_data:
        return {}

    distances = []
    functional_distances = []
    lexical_distances = []

    functional_deprels = {'det', 'case', 'aux', 'mark', 'cc', 'punct', 'aux:pass', 'aux:cop'}

    for sentence in treebank_data:
        # Build word index from sentence
        words = []
        for token in sentence:
            if 'id' in token and 'deprel' in token:
                words.append({
                    'id': int(token['id']),
                    'deprel': token.get('deprel', ''),
                    'head': int(token.get('head', 0)),
                })

        for word in words:
            if word['head'] == 0:
                continue  # Skip root
            dist = abs(word['id'] - word['head'])
            distances.append(dist)

            if word['deprel'] in functional_deprels or word['deprel'].startswith('aux'):
                functional_distances.append(dist)
            else:
                lexical_distances.append(dist)

    result = {
        'mean_mdd': float(np.mean(distances)) if distances else 0.0,
        'n_arcs': len(distances),
        'functional_mdd': float(np.mean(functional_distances)) if functional_distances else 0.0,
        'lexical_mdd': float(np.mean(lexical_distances)) if lexical_distances else 0.0,
    }

    # Distribution statistics
    if distances:
        result.update({
            'median_mdd': float(np.median(distances)),
            'std_mdd': float(np.std(distances)),
            'p90_mdd': float(np.percentile(distances, 90)),
            'min_mdd': int(np.min(distances)),
            'max_mdd': int(np.max(distances)),
        })

    # Entropy of distribution
    if distances and entr is not None:
        # Compute histogram
        unique, counts = np.unique(distances, return_counts=True)
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        entropy_values = entr(probs)
        result['mdd_entropy'] = float(np.sum(entropy_values))

    return result


def compute_phoneme_distances(treebank_data: List[Dict], avg_phonemes_per_word: float) -> Dict[str, float]:
    """Approximate phoneme-space dependency distance."""
    # In real implementation, we'd use epitran for G2P
    # Here we approximate: phoneme distance ~ word distance * avg_phonemes_per_word
    word_results = compute_dependency_distances(treebank_data)

    if not word_results or word_results['n_arcs'] == 0:
        return {}

    phoneme_factor = max(avg_phonemes_per_word, 2.0)  # Minimum 2 phonemes/word

    result = {
        'mean_ppad': word_results['mean_mdd'] * phoneme_factor,
        'functional_ppad': word_results['functional_mdd'] * phoneme_factor,
        'lexical_ppad': word_results['lexical_mdd'] * phoneme_factor,
        'phoneme_factor': phoneme_factor,
    }

    return result

## Phonological Density Estimation

Estimates phonological density from language family and typological features, using approximate values from PHOIBLE.

In [ ]:
def estimate_phonological_density(lang_code: str, lang_family: str) -> Dict[str, float]:
    """Estimate phonological density based on language family and typological features."""
    # Phoneme inventory sizes by language family (approximate, from PHOIBLE)
    family_inventory = {
        'Indo-European': 35.0,
        'Sino-Tibetan': 45.0,
        'Turkic': 30.0,
        'Uralic': 32.0,
        'Afroasiatic': 38.0,
        'Niger-Congo': 42.0,
        'Austronesian': 28.0,
        'Japonic': 25.0,
        'Koreanic': 24.0,
        'Dravidian': 55.0,
        'Tupian': 48.0,
        'Other': 35.0,
    }

    max_inventory = 70.0  # Approximate maximum across languages

    # Base phoneme count from family
    base_phonemes = family_inventory.get(lang_family, 35.0)

    # Adjust for specific languages (approximate values from PHOIBLE)
    language_adjustments = {
        'en': 44.0, 'de': 45.0, 'fr': 36.0, 'es': 24.0, 'it': 30.0,
        'nl': 32.0, 'sv': 40.0, 'ru': 55.0, 'pl': 50.0, 'cs': 55.0,
        'ja': 25.0, 'ko': 24.0, 'zh': 42.0, 'ar': 38.0, 'he': 32.0,
        'tr': 30.0, 'fi': 32.0, 'hu': 32.0, 'th': 44.0, 'vi': 36.0,
        'ms': 24.0, 'id': 26.0, 'hi': 40.0, 'bn': 38.0, 'sw': 45.0,
        'pt': 36.0, 'da': 36.0, 'no': 36.0, 'el': 35.0, 'bg': 40.0,
        'hr': 40.0, 'sk': 45.0, 'sl': 45.0, 'lt': 40.0, 'lv': 40.0,
        'et': 30.0, 'ka': 42.0, 'hy': 45.0, 'eu': 35.0, 'ha': 45.0,
        'yo': 45.0, 'ig': 45.0,
    }

    phoneme_count = language_adjustments.get(lang_code, base_phonemes)

    # Phonological density = normalized phoneme count
    phoneme_density = phoneme_count / max_inventory

    # Phonotactic complexity proxy (based on family)
    family_complexity = {
        'Sino-Tibetan': 0.9,
        'Dravidian': 0.85,
        'Niger-Congo': 0.8,
        'Afroasiatic': 0.75,
        'Indo-European': 0.6,
        'Uralic': 0.5,
        'Turkic': 0.55,
        'Austronesian': 0.6,
        'Japonic': 0.4,
        'Koreanic': 0.35,
        'Other': 0.5,
    }

    phonotactic_complexity = family_complexity.get(lang_family, 0.5)

    # Syllable richness proxy
    syllable_richness = phonotactic_complexity * 0.8 + phoneme_density * 0.2

    # Composite phonological density
    phon_density = (phoneme_density * 0.5 + phonotactic_complexity * 0.3 + syllable_richness * 0.2)

    return {
        'phoneme_count': round(phoneme_count, 1),
        'phoneme_density': round(phoneme_density, 3),
        'phonotactic_complexity': round(phonotactic_complexity, 3),
        'syllable_richness': round(syllable_richness, 3),
        'phon_density': round(phon_density, 3),
    }


def get_language_info(lang_code: str) -> Dict[str, str]:
    """Get typological information for a language."""
    # Word order typology (Greenberg)
    word_order_map = {
        'en': 'SVO', 'de': 'SVO', 'fr': 'SVO', 'es': 'SVO', 'it': 'SVO',
        'nl': 'SVO', 'sv': 'SVO', 'ru': 'SVO', 'pl': 'SVO', 'cs': 'SVO',
        'ja': 'SOV', 'ko': 'SOV', 'zh': 'SVO', 'ar': 'VSO', 'he': 'VSO',
        'tr': 'SOV', 'fi': 'SVO', 'hu': 'SOV', 'th': 'SVO', 'vi': 'SVO',
        'ms': 'SVO', 'id': 'SVO', 'hi': 'SOV', 'bn': 'SOV', 'sw': 'SVO',
        'pt': 'SVO', 'da': 'SVO', 'no': 'SVO', 'el': 'SVO', 'bg': 'SVO',
        'hr': 'SVO', 'sk': 'SVO', 'sl': 'SVO', 'lt': 'SVO', 'lv': 'SVO',
        'et': 'SVO', 'ka': 'SOV', 'hy': 'SOV', 'eu': 'SOV', 'ha': 'SVO',
        'yo': 'SVO', 'ig': 'SVO',
    }

    # Language family
    family_map = {
        'en': 'Indo-European', 'de': 'Indo-European', 'fr': 'Indo-European',
        'es': 'Indo-European', 'it': 'Indo-European', 'nl': 'Indo-European',
        'sv': 'Indo-European', 'ru': 'Indo-European', 'pl': 'Indo-European',
        'cs': 'Indo-European', 'da': 'Indo-European', 'no': 'Indo-European',
        'pt': 'Indo-European', 'el': 'Indo-European', 'bg': 'Indo-European',
        'hr': 'Indo-European', 'sk': 'Indo-European', 'sl': 'Indo-European',
        'lt': 'Indo-European', 'lv': 'Indo-European', 'hy': 'Indo-European',
        'ja': 'Japonic', 'ko': 'Koreanic', 'zh': 'Sino-Tibetan',
        'ar': 'Afroasiatic', 'he': 'Afroasiatic', 'ha': 'Afroasiatic',
        'tr': 'Turkic', 'fi': 'Uralic', 'hu': 'Uralic', 'et': 'Uralic',
        'th': 'Tai-Kadai', 'vi': 'Austroasiatic', 'ms': 'Austronesian',
        'id': 'Austronesian', 'hi': 'Indo-European', 'bn': 'Indo-European',
        'sw': 'Niger-Congo', 'yo': 'Niger-Congo', 'ig': 'Niger-Congo',
        'ka': 'Kartvelian', 'eu': 'Language Isolate',
    }

    return {
        'word_order': word_order_map.get(lang_code, 'SVO'),
        'language_family': family_map.get(lang_code, 'Other'),
    }

## Synthetic Treebank Generation

Generates synthetic dependency trees for testing. In production, real UD treebanks from `commul/universal_dependencies` would be used.

In [ ]:
def _generate_synthetic_treebank(n_sentences: int = 50, avg_words: int = 10, lang_code: str = 'en') -> List[Dict]:
    """Generate synthetic dependency trees for testing with realistic variation."""
    import random
    # Use language-specific seed so each language gets different tree structures
    random.seed(RANDOM_SEED + hash(lang_code))

    sentences = []
    for sent_id in range(n_sentences):
        n_words = random.randint(5, avg_words * 2)
        sentence = []
        for word_id in range(1, n_words + 1):
            # Generate a synthetic dependency arc
            if word_id == 1:
                head = 0  # Root
            else:
                # Bias towards leftward dependencies (typical in UD)
                head = random.randint(max(0, word_id - 5), word_id - 1)

            deprel = random.choice(['nsubj', 'obj', 'iobj', 'obl', 'acl', 'advmod',
                                    'case', 'det', 'aux', 'mark', 'conj', 'appos'])

            sentence.append({
                'id': word_id,
                'head': head,
                'deprel': deprel,
                'text': f'word{word_id}',
            })
        sentences.append(sentence)

    return sentences


def load_sample_treebanks(n_languages: int = 5) -> List[Dict]:
    """Load sample UD treebanks for evaluation."""
    # Target languages with good UD coverage - ordered for family diversity
    target_languages = [
        'en', 'ja', 'zh', 'ar', 'tr', 'fi', 'hi', 'sw', 'de', 'fr',
        'es', 'it', 'nl', 'sv', 'ru', 'pl', 'cs', 'ko', 'he', 'th',
        'vi', 'ms', 'id', 'bn', 'pt', 'da', 'no', 'el', 'bg', 'hu',
        'hr', 'sk', 'sl', 'lt', 'lv', 'et', 'ka', 'hy', 'eu', 'ha',
        'yo', 'ig',
    ]

    sample = []
    # Always use synthetic data for demo (datasets library may not be available)
    logger.info("Generating synthetic treebanks for demo")
    for lang_code in target_languages[:n_languages]:
        sample.append({
            'lang_code': lang_code,
            'data': _generate_synthetic_treebank(N_SENTENCES, AVG_WORDS, lang_code),
            'n_sentences': N_SENTENCES,
            'source': 'synthetic',
        })
    return sample

## Regression Analysis

Runs OLS and mixed-effects regression of MDD on phonological density, with word order as covariate.

In [ ]:
def run_mixed_effects_regression(languages_data: List[Dict]) -> Dict[str, Any]:
    """Run mixed-effects regression analysis."""
    if len(languages_data) < 3:
        logger.warning(f"Insufficient data for regression ({len(languages_data)} languages)")
        return {
            'model': 'insufficient_data',
            'n_languages': len(languages_data),
            'n_families': len(set(d.get('language_family', 'Other') for d in languages_data)),
        }

    # Prepare data
    phon_density = []
    mdd_word = []
    mdd_phoneme = []
    word_orders = []
    families = []

    for lang_data in languages_data:
        if 'phon_density' not in lang_data or 'mdd_word' not in lang_data:
            continue
        phon_density.append(lang_data['phon_density'])
        mdd_word.append(lang_data['mdd_word'])
        mdd_phoneme.append(lang_data.get('mdd_phoneme', lang_data['mdd_word'] * 4.0))
        word_orders.append(1 if lang_data['word_order'] == 'SOV' else 0)
        families.append(lang_data['language_family'])

    if len(phon_density) < 3:
        return {'model': 'insufficient_data', 'n_languages': len(phon_density)}

    phon_density = np.array(phon_density)
    mdd_word = np.array(mdd_word)
    mdd_phoneme = np.array(mdd_phoneme)
    word_orders = np.array(word_orders)
    families = np.array(families)

    # Create design matrix
    n = len(phon_density)
    design_matrix = np.column_stack([
        np.ones(n),  # Intercept
        phon_density,
        word_orders,
    ])

    result = {
        'model': 'MixedLM' if MixedLM is not None else 'OLS',
        'n_languages': n,
        'n_families': len(np.unique(families)),
        'families': list(np.unique(families)),
    }

    # Try mixed-effects model first
    if MixedLM is not None and len(np.unique(families)) > 2:
        try:
            model = MixedLM(mdd_word, design_matrix, groups=families)
            fitted = model.fit(reml=True)
            result['mixedlm_results'] = {
                'phon_density_coef': float(fitted.params[1]),
                'phon_density_pval': float(fitted.pvalues[1]),
                'word_order_coef': float(fitted.params[2]),
                'word_order_pval': float(fitted.pvalues[2]),
                'aic': float(fitted.aic),
                'bic': float(fitted.bic),
                'converged': bool(fitted.converged),
            }
            logger.info(f"MixedLM: phon_density coef = {result['mixedlm_results']['phon_density_coef']:.4f}, p = {result['mixedlm_results']['phon_density_pval']:.4f}")
        except Exception as e:
            logger.warning(f"MixedLM failed: {e}, falling back to OLS")
            result['mixedlm_results'] = {'error': str(e)}

    # OLS fallback
    try:
        X = np.column_stack([np.ones(n), phon_density, word_orders])
        beta = np.linalg.lstsq(X, mdd_word, rcond=None)[0]
        residuals = mdd_word - X @ beta
        mse = np.mean(residuals ** 2)
        se = np.sqrt(mse * np.diag(np.linalg.pinv(X.T @ X)))
        t_stats = beta / se
        p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-3)) if stats is not None else np.zeros(3)

        # R-squared
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((mdd_word - np.mean(mdd_word)) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

        result['ols_results'] = {
            'intercept': float(beta[0]),
            'phon_density_coef': float(beta[1]),
            'word_order_coef': float(beta[2]),
            'phon_density_pval': float(p_values[1]),
            'word_order_pval': float(p_values[2]),
            'r_squared': float(r_squared),
            'adjusted_r_squared': float(1 - (1 - r_squared) * (n - 1) / (n - 3)) if n > 3 else 0.0,
        }
        logger.info(f"OLS: phon_density coef = {result['ols_results']['phon_density_coef']:.4f}, p = {result['ols_results']['phon_density_pval']:.4f}, R² = {result['ols_results']['r_squared']:.4f}")
    except Exception as e:
        logger.error(f"OLS regression failed: {e}")
        result['ols_results'] = {'error': str(e)}

    # Spearman correlation (non-parametric)
    if stats is not None:
        try:
            corr_mdd, p_mdd = stats.spearmanr(phon_density, mdd_word)
            corr_ppad, p_ppad = stats.spearmanr(phon_density, mdd_phoneme)
            result['spearman'] = {
                'mdd_correlation': float(corr_mdd),
                'mdd_pvalue': float(p_mdd),
                'ppad_correlation': float(corr_ppad),
                'ppad_pvalue': float(p_ppad),
            }
            logger.info(f"Spearman: MDD rho = {corr_mdd:.4f}, p = {p_mdd:.4f}")
        except Exception as e:
            logger.warning(f"Spearman correlation failed: {e}")

    return result


def run_family_variance_analysis(languages_data: List[Dict]) -> Dict[str, Any]:
    """Analyze variance across language families."""
    if not languages_data:
        return {'n_families': 0}

    families_data = {}
    for lang in languages_data:
        family = lang.get('language_family', 'Other')
        if family not in families_data:
            families_data[family] = {'mdd': [], 'phon_density': [], 'languages': []}
        if 'mdd_word' in lang:
            families_data[family]['mdd'].append(lang['mdd_word'])
            families_data[family]['phon_density'].append(lang.get('phon_density', 0))
            families_data[family]['languages'].append(lang['lang_code'])

    results = {
        'n_families': len(families_data),
        'families': {},
    }

    for family, data in families_data.items():
        mdds = np.array(data['mdd'])
        phon_densities = np.array(data['phon_density'])

        family_result = {
            'n_languages': len(mdds),
            'mean_mdd': float(np.mean(mdds)) if len(mdds) > 0 else None,
            'std_mdd': float(np.std(mdds)) if len(mdds) > 1 else None,
            'mean_phon_density': float(np.mean(phon_densities)) if len(phon_densities) > 0 else None,
            'languages': data['languages'],
        }

        # Add min/max if we have enough data
        if len(mdds) > 0:
            family_result['min_mdd'] = float(np.min(mdds))
            family_result['max_mdd'] = float(np.max(mdds))
            family_result['median_mdd'] = float(np.median(mdds))

        results['families'][family] = family_result

    # Overall statistics
    all_mdds = [d['mdd_word'] for d in languages_data if 'mdd_word' in d]
    if all_mdds:
        results['overall'] = {
            'mean_mdd': float(np.mean(all_mdds)),
            'std_mdd': float(np.std(all_mdds)),
            'min_mdd': float(np.min(all_mdds)),
            'max_mdd': float(np.max(all_mdds)),
            'n_languages': len(all_mdds),
        }

    return results

## Run Evaluation Pipeline

Process each language: generate treebank, compute dependency distances, estimate phonological density, and run regression.

In [ ]:
logger.info("=" * 60)
logger.info("Starting Phonotactic-Dependency Distance Evaluation")
logger.info("=" * 60)
logger.info(f"Config: {N_LANGUAGES} languages, {N_SENTENCES} sentences, {AVG_WORDS} avg words")

# Load sample treebanks
treebanks = load_sample_treebanks(N_LANGUAGES)
logger.info(f"Loaded {len(treebanks)} treebanks")

# Process each language
languages_data = []
for tb in treebanks:
    lang_code = tb['lang_code']
    lang_info = get_language_info(lang_code)
    phon_info = estimate_phonological_density(lang_code, lang_info['language_family'])

    # Compute dependency distances
    dist_results = compute_dependency_distances(tb['data'])

    # Estimate phoneme distances
    avg_phonemes = phon_info['phoneme_count'] / 10.0  # Rough estimate
    ppad_results = compute_phoneme_distances(tb['data'], avg_phonemes)

    lang_data = {
        'lang_code': lang_code,
        'language_family': lang_info['language_family'],
        'word_order': lang_info['word_order'],
        'phon_density': phon_info['phon_density'],
        'phoneme_count': phon_info['phoneme_count'],
        'phonotactic_complexity': phon_info['phonotactic_complexity'],
        'mdd_word': dist_results.get('mean_mdd', 0.0),
        'mdd_phoneme': ppad_results.get('mean_ppad', 0.0),
        'mdd_functional': dist_results.get('functional_mdd', 0.0),
        'mdd_lexical': dist_results.get('lexical_mdd', 0.0),
        'n_arcs': dist_results.get('n_arcs', 0),
        'n_sentences': tb.get('n_sentences', 0),
        'data_source': tb.get('source', 'unknown'),
    }
    languages_data.append(lang_data)

    logger.info(f"  {lang_code}: MDD={lang_data['mdd_word']:.3f}, PhonDensity={lang_data['phon_density']:.3f}")

# Run regression analysis
logger.info("\nRunning regression analysis...")
regression_results = run_mixed_effects_regression(languages_data)

# Run family variance analysis
logger.info("\nRunning family variance analysis...")
family_results = run_family_variance_analysis(languages_data)

# Compute DLM strength metric (lower MDD = stronger minimization)
for lang in languages_data:
    if lang['mdd_word'] > 0:
        lang['dlm_strength'] = 1.0 / lang['mdd_word']
    else:
        lang['dlm_strength'] = 0.0

logger.info("Evaluation complete!")

## Results Visualization

Display key results in tables and plots.

In [ ]:
# Print summary table
print("\n" + "=" * 90)
print("PHONOTACTIC-DEPENDENCY DISTANCE EVALUATION RESULTS")
print("=" * 90)

# Language-level results
print("\n--- Language-Level Results ---")
print(f"{'Lang':<6} {'Family':<20} {'WO':<5} {'PhonDens':<10} {'MDD_word':<10} {'MDD_phon':<10} {'DLM_str':<10}")
print("-" * 90)
for lang in languages_data:
    print(f"{lang['lang_code']:<6} {lang['language_family']:<20} {lang['word_order']:<5} "
          f"{lang['phon_density']:<10.3f} {lang['mdd_word']:<10.3f} "
          f"{lang['mdd_phoneme']:<10.3f} {lang['dlm_strength']:<10.4f}")

# Aggregate metrics
print("\n--- Aggregate Metrics ---")
metrics_agg = {
    'mean_mdd_word_space': float(np.mean([l['mdd_word'] for l in languages_data])),
    'mean_mdd_phoneme_space': float(np.mean([l['mdd_phoneme'] for l in languages_data])),
    'mean_dlm_strength': float(np.mean([l['dlm_strength'] for l in languages_data])),
    'mean_phon_density': float(np.mean([l['phon_density'] for l in languages_data])),
    'n_languages': len(languages_data),
    'n_families': family_results.get('n_families', 0),
}
for key, val in metrics_agg.items():
    print(f"  {key}: {val:.4f}")

# Regression results
if 'ols_results' in regression_results:
    print("\n--- OLS Regression Results ---")
    ols = regression_results['ols_results']
    print(f"  Intercept: {ols['intercept']:.4f}")
    print(f"  PhonDensity coef: {ols['phon_density_coef']:.4f} (p={ols['phon_density_pval']:.4f})")
    print(f"  WordOrder coef: {ols['word_order_coef']:.4f} (p={ols['word_order_pval']:.4f})")
    print(f"  R²: {ols['r_squared']:.4f}")

if 'spearman' in regression_results:
    print("\n--- Spearman Correlation ---")
    sp = regression_results['spearman']
    print(f"  MDD vs PhonDensity: rho={sp['mdd_correlation']:.4f}, p={sp['mdd_pvalue']:.4f}")
    print(f"  PPAD vs PhonDensity: rho={sp['ppad_correlation']:.4f}, p={sp['ppad_pvalue']:.4f}")

# Family variance
if 'families' in family_results:
    print("\n--- Family-Level Variance ---")
    for family, fdata in family_results['families'].items():
        print(f"  {family}: n={fdata['n_languages']}, mean_MDD={fdata['mean_mdd']:.3f}, "
              f"mean_phonDens={fdata['mean_phon_density']:.3f}")

In [ ]:
# Visualization: MDD vs Phonological Density scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: MDD vs Phonological Density
ax1 = axes[0]
colors = {'SVO': 'blue', 'SOV': 'red', 'VSO': 'green'}
for lang in languages_data:
    wo = lang['word_order']
    ax1.scatter(lang['phon_density'], lang['mdd_word'],
                c=colors.get(wo, 'gray'), s=100, edgecolors='black', zorder=3,
                label=f"{lang['lang_code']} ({wo})")

# Add regression line if available
if 'ols_results' in regression_results:
    ols = regression_results['ols_results']
    x_vals = np.linspace(min(l['phon_density'] for l in languages_data),
                         max(l['phon_density'] for l in languages_data), 100)
    y_vals = ols['intercept'] + ols['phon_density_coef'] * x_vals
    ax1.plot(x_vals, y_vals, 'k--', alpha=0.5, label=f'OLS (R²={ols["r_squared"]:.3f})')

ax1.set_xlabel('Phonological Density')
ax1.set_ylabel('Mean Dependency Distance (words)')
ax1.set_title('MDD vs Phonological Density')
ax1.legend(fontsize=8, loc='best')
ax1.grid(alpha=0.3)

# Plot 2: DLM Strength by Language
ax2 = axes[1]
lang_codes = [l['lang_code'] for l in languages_data]
dlm_values = [l['dlm_strength'] for l in languages_data]
wo_colors = [colors.get(l['word_order'], 'gray') for l in languages_data]
bars = ax2.bar(lang_codes, dlm_values, color=wo_colors, edgecolor='black')
ax2.set_xlabel('Language')
ax2.set_ylabel('DLM Strength (1/MDD)')
ax2.set_title('Dependency Distance Minimization Strength')
ax2.grid(axis='y', alpha=0.3)

# Add word order legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor='black', label=wo) for wo, c in colors.items()]
ax2.legend(handles=legend_elements, title='Word Order', fontsize=8)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as 'evaluation_results.png'")

In [ ]:
# Additional visualization: Functional vs Lexical MDD comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(languages_data))
width = 0.35

func_mdd = [l['mdd_functional'] for l in languages_data]
lex_mdd = [l['mdd_lexical'] for l in languages_data]

bars1 = ax.bar(x - width/2, func_mdd, width, label='Functional MDD', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, lex_mdd, width, label='Lexical MDD', color='coral', edgecolor='black')

ax.set_xlabel('Language')
ax.set_ylabel('Mean Dependency Distance')
ax.set_title('Functional vs Lexical Dependency Distance')
ax.set_xticks(x)
ax.set_xticklabels([l['lang_code'] for l in languages_data])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('func_vs_lex_mdd.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as 'func_vs_lex_mdd.png'")

In [ ]:
# Final summary: save results to JSON
output = {
    'metrics_agg': metrics_agg,
    'regression_results': regression_results,
    'family_results': family_results,
    'languages_data': languages_data,
    'config': {
        'n_languages': N_LANGUAGES,
        'n_sentences': N_SENTENCES,
        'avg_words': AVG_WORDS,
        'random_seed': RANDOM_SEED,
    }
}

with open('demo_results.json', 'w') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Results saved to 'demo_results.json'")
print(f"\nSummary: {len(languages_data)} languages evaluated across {family_results.get('n_families', 0)} families")
print(f"Mean MDD (word): {metrics_agg['mean_mdd_word_space']:.3f}")
print(f"Mean MDD (phoneme): {metrics_agg['mean_mdd_phoneme_space']:.3f}")
print(f"Mean DLM strength: {metrics_agg['mean_dlm_strength']:.4f}")